# PaliGemma-3B: replication gate and T viability (D76)

The second model (plan 3.3, D11): `google/paligemma2-3b-mix-448`. Everything
here is fixed in **DECISIONS.md D76**, written before this notebook existed.

**Method, from Lietzow's repo (commit 145b453):** the same prompts as Qwen,
passed as plain text (PaliGemma has no chat template), 4-bit nf4, and the same
layer path. Their patching and knockout run **a full forward pass per patched
head**, with the patch at the last position. We do the same
(`paligemma.FullForwardRunner`), not Qwen's cached speed-up, because
PaliGemma's image + prompt prefix may attend both ways; section 3 measures it.

**What it produces** (in `/kaggle/working/pg_replication/`):

1. `pg_selection_paligemma2_3b.json`: every object's top-1 under the Prior and
   Visual prompts (P) and with a neutral and an incongruent note (T). Gives
   the P correctly-conflicting set (Lietzow: 121) and the **T viability gate**
   (D76 (6): at least 50 T-conflicting objects).
2. `pg_replication_knockout_paligemma2_3b.json`: pairwise knockout (D67) of
   their 8 promoting and 2 suppressing heads, as a group and one at a time.
3. `pg_replication_patching_paligemma2_3b.json`: patching over all heads on
   the P objects, and our classification vs theirs.

Download all three into `results/` and run
`scripts/analyze_pg_replication.py` for the D76 verdict.

**Before running:** the PaliGemma 2 checkpoints are gated. The Hugging Face
account behind your `HF_TOKEN` must have accepted Google's licence on the
model page of `google/paligemma2-3b-mix-448`, or the load fails with a 403.

**Untested:** none of this has run on the real model. Section 3 stops the run
if the prompt, the image grid or the hooks are wrong. Run it as **Save Version
-> Save & Run All**; every loop checkpoints per object (D44) and resumes from
an attached previous output. Settings: GPU T4, Internet on, `HF_TOKEN` ticked.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/SohamRShinde/VLM-Overlap-RP"
REPO_DIR = "/kaggle/working/research_paper"

import importlib, os, subprocess, sys

if REPO_URL:
    if os.path.exists(REPO_DIR):
        # Always update an existing checkout: a stale clone silently runs old
        # code, and the failure shows up later as a confusing ImportError.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "-q", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# Drop any already-imported package modules: Python caches them, so after an
# update the kernel would keep serving the old file.
for _m in [m for m in list(sys.modules) if m.startswith("circuit_overlap")]:
    del sys.modules[_m]
importlib.invalidate_caches()

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("repo at", COMMIT)   # record this in RESULTS.md with any number from this run

# Fail here, loudly, if the checkout is older than this notebook expects.
import importlib.util as _ilu
_need = {}
for _mod, _names in {
    "circuit_overlap.paligemma": ["CHECKPOINT", "load_paligemma", "build_inputs", "text_geometry",
                                  "FullForwardRunner", "square_image_block",
                                  "last_token_reaches_earlier_positions"],
    "circuit_overlap.patching": ["patch_example", "mean_restoration_grids", "classify_heads",
                                 "rank_heads", "capture_last_token_o_proj_inputs",
                                 "patch_head_last_token"],
    "circuit_overlap.cross_ablation": ["ablate_heads_last_token"],
    "circuit_overlap.replication": ["lietzow_rows", "is_correctly_conflicting",
                                    "knockout_flip_counts", "run_cached"],
    "circuit_overlap.t_patching": ["render_t_pair", "is_t_conflicting", "LAYOUT"],
    "circuit_overlap.t_screen": ["PRIOR_PROMPT", "VISUAL_PROMPT", "color_first_token_ids",
                                 "load_notes_config", "native_bf16"],
    "circuit_overlap.heads": ["load_head_set", "format_head_label"],
    "circuit_overlap.metrics": ["containment", "jaccard_permutation_test"],
    "circuit_overlap.provenance": ["result_envelope", "write_result"],
}.items():
    if _ilu.find_spec(_mod) is None:
        _need[_mod] = ["module"]
        continue
    _m = importlib.import_module(_mod)
    _gone = [n for n in _names if not hasattr(_m, n)]
    if _gone:
        _need[_mod] = _gone
for _f in ("configs/heads_lietzow_paligemma2_3b.json", "scripts/analyze_pg_replication.py"):
    if not os.path.exists(f"{REPO_DIR}/{_f}"):
        _need[_f] = ["file"]
if _need:
    raise RuntimeError(f"checkout {COMMIT} is missing {_need}. "
                       "Push the local commits to origin/main, then re-run this cell.")
print("code is current for this notebook")

In [ ]:
# Use one GPU. On "GPU T4 x2", device_map="auto" splits the model across both
# cards, which only adds transfer time for a 3B model. This must run before
# torch is first imported, so restart the kernel if torch is already loaded.
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# bitsandbytes is required for the 4-bit load (D10). Installing it into a
# kernel that already imported transformers does nothing: restart and re-run.
!pip install -q -U "transformers>=4.49" accelerate datasets inflect "bitsandbytes>=0.46.1"
import torch, transformers
from transformers.utils import is_bitsandbytes_available
from circuit_overlap.t_screen import native_bf16
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
# Not torch.cuda.is_bf16_supported(): it says True on a T4 (emulation).
print("bitsandbytes available:", is_bitsandbytes_available(),
      "| native bf16:", native_bf16(torch.cuda.get_device_capability(0)))

from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    # Secrets are attached PER NOTEBOOK: one on your account is not enough.
    raise RuntimeError(
        "HF_TOKEN is not attached to this notebook. In the editor: Add-ons -> "
        "Secrets -> tick the checkbox next to HF_TOKEN (exact label), then "
        "restart the session and run from cell 1. It must be ticked before "
        "Save & Run All, or the background run fails here."
    ) from exc

## 2. Model, geometry, and Lietzow's heads

`text_geometry` reads layers, heads and `head_dim` from the config and checks
them against `o_proj`'s input width. Gemma-2 sets `head_dim` explicitly and it
is not hidden_size / n_heads, so the fallback the Qwen notebooks use would be
wrong here.

In [ ]:
import json
from circuit_overlap.paligemma import CHECKPOINT, load_paligemma, text_geometry
from circuit_overlap.heads import load_head_set, format_head_label
from circuit_overlap.t_screen import PRIOR_PROMPT, VISUAL_PROMPT

model, processor = load_paligemma()
COMPUTE_DTYPE = str(model.config.quantization_config.bnb_4bit_compute_dtype).replace("torch.", "")
print(CHECKPOINT, "| compute dtype:", COMPUTE_DTYPE, "(D76 (3): fp16 on a T4)")
print("devices used:", sorted({str(d) for d in getattr(model, "hf_device_map", {"": model.device}).values()}))

geo = text_geometry(model)          # raises if o_proj's width != n_heads x head_dim
n_layers, n_heads, head_dim = geo["n_layers"], geo["n_heads"], geo["head_dim"]
print(geo)

HEADS_FILE = f"{REPO_DIR}/configs/heads_lietzow_paligemma2_3b.json"
hs = load_head_set(HEADS_FILE, n_layers=n_layers, n_heads_per_layer=n_heads)   # raises if out of range
hs_raw = json.load(open(HEADS_FILE))
print("their promoting :", [format_head_label(h) for h in hs.promoting])
print("their suppressing:", [format_head_label(h) for h in hs.suppressing])
print("they report:", hs_raw["group_ablation_flip_pct"], "on", hs_raw["n_correctly_conflicting"],
      "objects; single heads", hs.single_head_prior_flip_pct)

## 3. [VERIFY] prompt, image grid, attention direction, hooks, speed

Stops the notebook if any of these is wrong:

1. the text PaliGemma reads after the image is the prompt (printed), and the
   last token is text;
2. the image tokens are one contiguous square block (expected 32 x 32 = 1,024
   for a 448-pixel model);
3. the colour words have distinct first tokens on the Gemma tokenizer (D26);
4. an empty ablation changes nothing, ablating their promoting heads changes
   the logits, and every layer's captured head outputs are n_heads x head_dim;
5. the two T renders share a footprint and a length (single-factor contrast).

It also measures whether changing only the **last** token changes the final
state of earlier positions (`last_token_reaches_earlier_positions`). If it
does, the prefix attends both ways and Qwen's cached path would have been
wrong here; the full forward pass used below is right either way (D76 (2)).

In [ ]:
import glob, inflect, pathlib, shutil, time
import torch
from collections import Counter
from IPython.display import display
from datasets import load_dataset
from circuit_overlap.paligemma import (
    FullForwardRunner, build_inputs as _pg_inputs, square_image_block,
    last_token_reaches_earlier_positions,
)
from circuit_overlap.patching import capture_last_token_o_proj_inputs
from circuit_overlap.cross_ablation import ablate_heads_last_token
from circuit_overlap.replication import lietzow_rows
from circuit_overlap.t_screen import color_first_token_ids, load_notes_config
from circuit_overlap.t_patching import LAYOUT, render_t_pair

WORK = pathlib.Path("/kaggle/working/pg_replication")
CKPT = WORK / "ckpt"
CKPT.mkdir(parents=True, exist_ok=True)
# Resume: bring back checkpoints from an attached previous run's output.
for name in ("select.jsonl", "knockout.jsonl", "patching.jsonl"):
    found = sorted(glob.glob(f"/kaggle/input/**/pg_replication/ckpt/{name}", recursive=True))
    if found and not (CKPT / name).exists():
        shutil.copy(found[0], CKPT / name)
        print(f"resuming from {found[0]}")

ds = load_dataset("mgolov/Visual-Counterfact", split="color")
_eng = inflect.engine()
rows, dropped = lietzow_rows(ds, _eng)
print(len(rows), "objects", dict(dropped), "(the Qwen runs used the same 469)")
vocab = sorted({c for r in rows for c in r["correct"]} | {r["attack"] for r in rows})
color_ids, collisions = color_first_token_ids(processor.tokenizer, vocab)
ambiguous = set(collisions)
print(len(vocab), "colours | sharing a first token on the Gemma tokenizer (D26):", collisions)

def build_inputs(image, prompt):
    return _pg_inputs(processor, image, prompt, model.device)

def prompts(r):
    return (VISUAL_PROMPT.format(object=r["name"]),
            PRIOR_PROMPT.format(a_object=_eng.a(r["name"])))

notes_cfg = load_notes_config(f"{REPO_DIR}/configs/notes.json")
spec = next(s for s in notes_cfg["specs"] if s.name == LAYOUT)
neutral_words = notes_cfg["neutral_words"]

def t_inputs(r):
    """(incongruent inputs, neutral inputs, render info), as t_patching.ipynb."""
    row = ds[r["i"]]
    pr = render_t_pair(row["original_image"], row["counterfact_image"], r["attack"], neutral_words, spec)
    prompt = VISUAL_PROMPT.format(object=r["name"])
    return build_inputs(pr["incongruent"], prompt), build_inputs(pr["neutral"], prompt), pr

VERIFY = {"compute_dtype": COMPUTE_DTYPE, "geometry": geo, "ambiguous_colours": sorted(ambiguous)}
IMAGE_TOKEN_ID = processor.image_token_id      # their get_image_token_id

# [VERIFY] 1-2. the prompt and the image block.
r0 = rows[0]
inp0 = build_inputs(ds[r0["i"]]["counterfact_image"], prompts(r0)[1])
ids0 = inp0["input_ids"][0].tolist()
offset0, side0 = square_image_block(ids0, IMAGE_TOKEN_ID)
text0 = processor.tokenizer.decode(ids0[offset0 + side0 * side0:])
VERIFY["image_block"] = {"offset": offset0, "side": side0, "n_tokens": len(ids0),
                         "pixel_values": list(inp0["pixel_values"].shape)}
VERIFY["prompt_after_image"] = text0
print("inputs:", {k: tuple(v.shape) for k, v in inp0.items()})
print("image block:", VERIFY["image_block"])
print("text after the image:", repr(text0))
assert ids0[-1] != IMAGE_TOKEN_ID, "the last token must be text"
assert prompts(r0)[1] in text0, "the prompt is not what the model reads"

# [VERIFY] 3. colour tokens: colliding colours are excluded (D26), and reported.
print("excluded colours (D26):", sorted(ambiguous) or "none")

# [VERIFY] 4. hooks in a full forward pass.
runner0 = FullForwardRunner(model, inp0)
base0 = runner0.logits()
with ablate_heads_last_token(model, [], head_dim):
    same0 = runner0.logits()
with ablate_heads_last_token(model, hs.promoting, head_dim):
    knocked0 = runner0.logits()
with capture_last_token_o_proj_inputs(model) as acts0:
    runner0.logits()
VERIFY["empty_ablation_max_dlogit"] = float((same0 - base0).abs().max())
VERIFY["promoting_ablation_max_dlogit"] = float((knocked0 - base0).abs().max())
print("empty ablation max |dlogit|:", VERIFY["empty_ablation_max_dlogit"], "(expect ~0)")
print("promoting ablation max |dlogit|:", VERIFY["promoting_ablation_max_dlogit"], "(expect > 0)")
assert VERIFY["empty_ablation_max_dlogit"] < 1e-3
assert VERIFY["promoting_ablation_max_dlogit"] > 1e-3
assert sorted(acts0) == list(range(n_layers)) and all(
    acts0[l].shape[-1] == n_heads * head_dim for l in acts0), "captured head outputs have the wrong shape"
del acts0

# [VERIFY] 5. the T renders.
s0_, n0_, pr0 = t_inputs(r0)
print(f"{r0['name']}: note {r0['attack']!r} vs neutral {pr0['neutral_word']!r}")
display(pr0["neutral"].resize((256, 256))); display(pr0["incongruent"].resize((256, 256)))
assert pr0["same_note_footprint"], "the two notes differ in shape: not a single-factor contrast"
assert s0_["input_ids"].shape == n0_["input_ids"].shape, "renders tokenise to different lengths"

# Attention direction (reported, D76 (2)).
VERIFY["last_token_vs_position_0"] = last_token_reaches_earlier_positions(model, inp0, position=0)
VERIFY["last_token_vs_first_text"] = last_token_reaches_earlier_positions(
    model, inp0, position=offset0 + side0 * side0)
for k in ("last_token_vs_position_0", "last_token_vs_first_text"):
    print(k, VERIFY[k])

# Speed, and the total, before any long loop (CLAUDE.md).
for attempt in ("warm-up", "timed"):
    torch.cuda.synchronize(); t0 = time.time()
    runner0.logits()
    torch.cuda.synchronize()
    step = time.time() - t0
    print(f"{attempt} forward pass: {step:.2f}s")
VERIFY["forward_s"] = step
n_conf = hs_raw["n_correctly_conflicting"]          # their count, only for this estimate
n_knock = 1 + 2 + len(hs.promoting) + len(hs.suppressing)
print(f"estimate: selection ~{len(rows) * 4 * step / 60:.0f} min, knockout ~{n_conf * 2 * n_knock * step / 60:.0f} min, "
      f"patching ~{n_conf * (2 + 2 * n_layers * n_heads) * step / 3600:.1f} h (at {n_conf} P objects)")
del runner0, base0, same0, knocked0

## 4. Selection: P and T (D76 (4), (6))

Per object, four full forward passes: Prior and Visual prompts on the
counterfactual image (P), and the Visual prompt on the original image with a
neutral and an incongruent `on_object_small` note (T). Non-finite logits drop
the object (plan risk 7). Saves `pg_selection_paligemma2_3b.json` at once, so
the T viability count survives even if a later loop dies.

In [ ]:
from circuit_overlap.replication import is_correctly_conflicting, run_cached
from circuit_overlap.t_patching import is_t_conflicting
from circuit_overlap.provenance import result_envelope, write_result

def _finite(t):
    return bool(torch.isfinite(t).all())

def select_object(r):
    cf = ds[r["i"]]["counterfact_image"]
    visual_prompt, prior_prompt = prompts(r)
    p_in, v_in = build_inputs(cf, prior_prompt), build_inputs(cf, visual_prompt)
    inc_in, neu_in, _ = t_inputs(r)
    lp, lv = FullForwardRunner(model, p_in).logits(), FullForwardRunner(model, v_in).logits()
    li, ln = FullForwardRunner(model, inc_in).logits(), FullForwardRunner(model, neu_in).logits()
    out = {"name": r["name"], "correct": r["correct"], "attack": r["attack"]}
    if not all(map(_finite, (lp, lv, li, ln))):
        return {**out, "status": "numerics_failed"}
    out.update(status="ok", top1_prior=int(lp.argmax()), top1_visual=int(lv.argmax()),
               top1_incongruent=int(li.argmax()), top1_neutral=int(ln.argmax()),
               p_same_length=p_in["input_ids"].shape == v_in["input_ids"].shape,
               t_same_length=inc_in["input_ids"].shape == neu_in["input_ids"].shape)
    out["p_conflicting"] = is_correctly_conflicting(r, out["top1_visual"], out["top1_prior"], color_ids, ambiguous)
    out["t_conflicting"] = is_t_conflicting(r, out["top1_neutral"], out["top1_incongruent"], color_ids, ambiguous)
    return out

select = run_cached(CKPT / "select.jsonl", rows, select_object, progress_every=25)
sel_by_i = {s["i"]: s for s in select}
p_rows = [r for r in rows if sel_by_i[r["i"]].get("p_conflicting")]
t_rows = [r for r in rows if sel_by_i[r["i"]].get("t_conflicting")]
print(Counter(s["status"] for s in select))
print(f"P correctly conflicting: {len(p_rows)}  (Lietzow: {hs_raw['n_correctly_conflicting']})")
print(f"T conflicting: {len(t_rows)}  (D76 (6) gate: >= 50)  | both: {len({r['i'] for r in p_rows} & {r['i'] for r in t_rows})}")

def _envelope(experiment, payload, n):
    return result_envelope(
        experiment, payload, model="PaliGemma2-3B-mix-448", checkpoint=CHECKPOINT, seed=0,
        prompt=f"visual={VISUAL_PROMPT!r} prior={PRIOR_PROMPT!r}; T: visual, {LAYOUT} note",
        n=n, repo_dir=REPO_DIR, decision="D76", quantisation="4-bit nf4, double quant",
        compute_dtype=COMPUTE_DTYPE, gpu=torch.cuda.get_device_name(0),
        image_tokens=side0 * side0, method="full forward pass per intervention (Lietzow)")

print(write_result(WORK / "pg_selection_paligemma2_3b.json", _envelope(
    "pg_selection", {"n_objects": len(rows), "dropped": dict(dropped), "color_ids": color_ids,
                     "ambiguous_colours": sorted(ambiguous), "verify": VERIFY, "records": select},
    len(rows))))

## 5. Knockout of their heads (D76 (5) a, b, f)

Pairwise flips (D67): each P object's Prior and Visual runs, with each of
their 10 classified heads zeroed alone, their 8 promoting together, and their
2 suppressing together, recording the logits of the two runs' unablated
answers. Same record format as `replication_flip_rate.ipynb`, so
`replication.knockout_flip_counts` and the gate script read it unchanged.

In [ ]:
from circuit_overlap.replication import knockout_flip_counts

configs = {"grp_promoting": list(hs.promoting), "grp_suppressing": list(hs.suppressing)}
for h in list(hs.promoting) + list(hs.suppressing):
    configs[f"ind_{format_head_label(h)}"] = [h]

def knockout_pairs(r):
    s = sel_by_i[r["i"]]
    a_prior, a_visual = s["top1_prior"], s["top1_visual"]
    cf = ds[r["i"]]["counterfact_image"]
    visual_prompt, prior_prompt = prompts(r)
    runs = {"prior": FullForwardRunner(model, build_inputs(cf, prior_prompt)),
            "visual": FullForwardRunner(model, build_inputs(cf, visual_prompt))}
    out = {"name": r["name"], "a_prior": a_prior, "a_visual": a_visual}
    for name, heads in [("none", [])] + list(configs.items()):
        for kind, run in runs.items():
            with ablate_heads_last_token(model, heads, head_dim):
                logits = run.logits()
            out[f"{name}/{kind}"] = [round(float(logits[a_prior]), 4), round(float(logits[a_visual]), 4)]
    return out

knock = run_cached(CKPT / "knockout.jsonl", p_rows, knockout_pairs, progress_every=10)
counts = {name: knockout_flip_counts(knock, name) for name in ["none", *configs]}
print("sanity, no knockout (expect 0 and 0):", counts["none"]["prior_flips"], counts["none"]["visual_flips"])
for name in ("grp_promoting", "grp_suppressing"):
    c = counts[name]
    print(f"{name:16s} prior flip {c['prior_flip_rate']:6.1%}   visual flip {c['visual_flip_rate']:6.1%}")
print("they report, grp_promoting:", hs_raw["group_ablation_flip_pct"])

print(write_result(WORK / "pg_replication_knockout_paligemma2_3b.json", _envelope(
    "pg_replication_knockout",
    {"flip_definition": "pairwise, Lietzow knockout_attn_heads (D67)",
     "configs": {k: [list(h) for h in v] for k, v in configs.items()},
     "counts": counts, "records": knock,
     "reported": {"grp_promoting": {"prior_flip": hs_raw["group_ablation_flip_pct"]["prior"] / 100,
                                    "visual_flip": hs_raw["group_ablation_flip_pct"]["visual"] / 100},
                  "single_head_prior_flip_pct": hs.single_head_prior_flip_pct,
                  "n": hs_raw["n_correctly_conflicting"]}},
    len(knock))))

## 6. Patching sweep over every head (D76 (5) c, d, e)

For each P object, both directions over all heads with `patch_example`, using
full forward passes. The longest loop by far: section 3 printed the estimate.
An object whose Prior and Visual prompts tokenise to different lengths is
recorded and skipped, as Lietzow do.

In [ ]:
from circuit_overlap.patching import patch_example

def sweep_object(r):
    cf = ds[r["i"]]["counterfact_image"]
    visual_prompt, prior_prompt = prompts(r)
    source_in, target_in = build_inputs(cf, prior_prompt), build_inputs(cf, visual_prompt)
    if source_in["input_ids"].shape != target_in["input_ids"].shape:
        return {"status": "shape_mismatch", "name": r["name"]}
    record = patch_example(model, FullForwardRunner(model, source_in), FullForwardRunner(model, target_in), head_dim)
    if record is None:
        return {"status": "same_argmax", "name": r["name"]}
    for key, value in record.items():          # 4 decimals keeps the file small
        if isinstance(value, list):
            record[key] = [[round(x, 4) for x in row] for row in value]
    return {"status": "patched", "name": r["name"], **record}

sweep = run_cached(CKPT / "patching.jsonl", p_rows, sweep_object, progress_every=5)
print(Counter(s["status"] for s in sweep))

## 7. Classify, compare with their list, save

Lietzow's PCA +/- 2 sigma rule (`classify_heads`). The verdict is applied by
`scripts/analyze_pg_replication.py` to the three saved files; the printout at
the end is a **preliminary** run of that same script.

In [ ]:
import sys
import numpy as np
from circuit_overlap.patching import mean_restoration_grids, classify_heads, rank_heads
from circuit_overlap.metrics import containment, jaccard_permutation_test

patched = [s for s in sweep if s["status"] == "patched"]
p2v, v2p, n_used = mean_restoration_grids(patched)
cls = classify_heads(p2v, v2p, sigma=2.0)
ranks = rank_heads(cls["pc1_scores"])
ours_p, theirs_p = set(cls["promoting"]), set(hs.promoting)
print(f"{n_used} objects scored | our promoting:", sorted(format_head_label(h) for h in ours_p))
print("our suppressing:", sorted(format_head_label(h) for h in cls["suppressing"]))
print(f"their promoting found: {len(ours_p & theirs_p)}/{len(theirs_p)}, containment {containment(theirs_p, ours_p):.2f}")
print("PC1 ranks of theirs:", {format_head_label(h): ranks[h] for h in sorted(theirs_p)})

print(write_result(WORK / "pg_replication_patching_paligemma2_3b.json", _envelope(
    "pg_replication_patching",
    {"n_objects": len(rows), "status_counts": dict(Counter(s["status"] for s in sweep)),
     "n_scored": n_used, "p2v_grid": p2v.round(6).tolist(), "v2p_grid": v2p.round(6).tolist(),
     "pc1_scores": cls["pc1_scores"].round(6).tolist(), "pc1_direction": cls["pc1_direction"],
     "pc1_mean": cls["pc1_mean"], "pc1_std": cls["pc1_std"], "sigma": cls["sigma"],
     "promoting": [list(h) for h in sorted(ours_p)],
     "suppressing": [list(h) for h in sorted(cls["suppressing"])],
     "reference_promoting": [list(h) for h in sorted(theirs_p)],
     "reference_suppressing": [list(h) for h in sorted(hs.suppressing)],
     "records": patched},
    n_used)))

sys.path.insert(0, f"{REPO_DIR}/scripts")
from analyze_pg_replication import main as d76_gate
print("\nPRELIMINARY (the committed script on this run's files):\n")
d76_gate(str(WORK / "pg_selection_paligemma2_3b.json"),
         str(WORK / "pg_replication_knockout_paligemma2_3b.json"),
         str(WORK / "pg_replication_patching_paligemma2_3b.json"))
print("\nDownload the three files into results/ and run scripts/analyze_pg_replication.py.")